In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import spacy
import json

model_id = "Zual/MPropositioneur-V2-large"

tokenizer = AutoTokenizer.from_pretrained(model_id)
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16).to(device)

nlp = spacy.load("en_core_web_sm")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [9]:
def display(text):
    doc = nlp(text)
    spacy.displacy.render(doc, style = "dep")

def atomize(text):
    prompt = f"<|im_start|>user\nAtomize: {text}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=8192).to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=2048, do_sample=False)

    generated_ids = outputs[0][inputs.input_ids.shape[1]:]


    result = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # La sortie est une liste JSON : ["p1", "p2", ...]
    propositions = json.loads(result)

    return propositions

In [10]:
sentence = "Luc is working on a project with his interns, Younes and Ava. They all work at LISN."

In [12]:
atomic_props = atomize(sentence)
atomic_props

['Luc is working on a project.',
 'Luc is working on the project with his interns.',
 "Luc's interns are Younes and Ava.",
 'Luc, Younes, and Ava all work at LISN.']

In [13]:
display(sentence)

In [15]:
for p in atomic_props:
    display(p)